# BlueBird Flexibility Manager — Code Walkthrough

This notebook walks through every component built/modified in the dockerization session.
Work through each section to own the code.

**System overview:**
```
dummy-forecast  →  writes disturbance forecasts (T_ambient, door_load) to SQLite
fc (FlexOPTi)   →  reads forecasts + DT model + TM prices → runs MPC → writes setpoints
control-room    →  Flask dashboard showing service health + DB contents
dt-ewh          →  Digital Twin (R), run on-demand to retrain dynamics model
```

**Shared state:**
- `db-data` volume → `/data/bb.db` (SQLite, shared by all Python services + FC)
- `dt-volume` → `/dt-volume/cold_room_model_continuous.json` (DT writes, FC reads)

---
## 1. Database Schema (`db/init.sql`)

SQLite schema shared by all services. Created by `dummy-forecast` on startup.

In [1]:
# Read and print the schema
with open('db/init.sql') as f:
    print(f.read())

-- BlueBird shared database schema
-- Engine-agnostic DDL (compatible with SQLite and PostgreSQL)

-- Sensor measurements written by the plant / BMS
CREATE TABLE IF NOT EXISTS measurements (
    id        INTEGER PRIMARY KEY,
    timestamp TEXT    NOT NULL,
    room_id   INTEGER NOT NULL,
    T_air     REAL    NOT NULL,
    T_product REAL    NOT NULL
);

-- Disturbance forecasts written by the Disturbances Forecast service
CREATE TABLE IF NOT EXISTS disturbance_forecasts (
    id        INTEGER PRIMARY KEY,
    timestamp TEXT    NOT NULL,
    room_id   INTEGER NOT NULL,
    T_ambient REAL    NOT NULL,
    door_load REAL    NOT NULL DEFAULT 0.0
);

-- Control setpoints written by the Flexibility Controller, read by the Knowledge Engine
CREATE TABLE IF NOT EXISTS setpoints (
    id        INTEGER PRIMARY KEY,
    timestamp TEXT    NOT NULL,
    room_id   INTEGER NOT NULL,
    u         REAL    NOT NULL,
    p3        REAL    NOT NULL DEFAULT 0.0
);

-- NeuralForecast outputs (existing sc

**Key tables:**
| Table | Writer | Reader |
|---|---|---|
| `disturbance_forecasts` | dummy-forecast | fc (FlexOPTi) |
| `measurements` | fc | control-room |
| `setpoints` | fc | control-room |
| `assets` | init seed | fc |

**Why SQLite?** Zero infra. File on shared Docker volume. Fine for single-host demo.

---
## 2. Dummy Forecast Service (`forecasting/dummy/`)

Pure Python, no extra deps. Two jobs:
1. Initialize DB schema + seed assets on startup
2. Write synthetic disturbance forecasts every `CADENCE_S` seconds

In [2]:
with open('forecasting/dummy/dummy_forecast.py') as f:
    print(f.read())

"""
Dummy Disturbances Forecast service.

Writes synthetic disturbance forecasts to the database on a configurable cadence.
Interface-compatible with the real NeuralForecast service: writes rows to the
`disturbance_forecasts` table with columns (timestamp, room_id, T_ambient, door_load).

Reference pattern: FlexOPTi/scripts/ewh/mpc_base.jl (constant 18°C ambient, zero door load).

Environment variables:
  DB_PATH              path to the SQLite database file
  DB_FORECASTS_TABLE   table name for disturbance forecasts (default: disturbance_forecasts)
  CADENCE_S            seconds between forecast updates (default: 900)
  HORIZON_STEPS        number of forecast steps to write (default: 48)
  DELTA_T_S            seconds per step (default: 900)
  N_ROOMS              number of rooms (default: 5)
  T_AMBIENT            constant ambient temperature in °C (default: 18.0)
"""

import os
import sqlite3
import time
import logging
import threading
from datetime import datetime, timedelta, timez

**What to notice:**
- `init_db()` — creates tables + inserts 5 cold-room assets if not present
- `write_forecast()` — loops over rooms, inserts rows with `T_ambient=18.0, door_load=0.0`
- Health endpoint on port 5000 — used by control-room to check liveness
- No Flask, no pandas — uses `sqlite3` (stdlib) + `http.server` (stdlib) in a thread

**To extend:** Replace hardcoded 18°C with a real weather API call.

In [3]:
with open('forecasting/dummy/Dockerfile') as f:
    print(f.read())

FROM python:3.12-slim

WORKDIR /app

COPY dummy_forecast.py .

CMD ["python", "-u", "dummy_forecast.py"]



---
## 3. FlexOPTi Julia Package — What Changed

### 3a. Package manifest (`FlexOPTi/Project.toml`)

Removed heavy/unused packages to cut Docker build time from ~40min to ~10min:

In [4]:
with open('FlexOPTi/Project.toml') as f:
    print(f.read())

name = "FlexOPTi"
uuid = "ddc234f0-be43-11f0-22b0-d31c67020283"
authors = ["kahka"]

[deps]
DBInterface = "a10d1c49-ce27-4219-8d33-6db1a4562965"
Dates = "ade2ca70-3891-5945-98fb-dc099432e06a"
DrWatson = "634d3b9d-ee7a-5ddf-bec9-22491ea816e1"
HTTP = "cd3eb016-35fb-5094-929b-558a96fad6f3"
HiGHS = "87dc4568-4c63-4d18-b0c0-bb2238e4078b"
JLD2 = "033835bb-8acc-5ee8-8aae-3f567f8a3819"
JSON = "682c06a0-de6a-54ab-a142-c8b1cf79cde6"
JSON3 = "0f8b85d8-7281-11e9-16c2-39a750bddbf1"
JuMP = "4076af6c-e467-56ae-b986-b466b2749572"
LinearAlgebra = "37e2e46d-f89d-539d-b4ee-838fcccc9c8e"
Logging = "56ddb016-857b-54e1-b83d-db4d58db5568"
LoggingExtras = "e6f89c97-d47a-5376-807f-9c37f3926c36"
MathOptInterface = "b8f27783-ece8-5eb3-8dc8-9495eed66fee"
Printf = "de0858da-6303-5e67-8744-51eddeeeb8d7"
Random = "9a3f8284-a2c9-5f02-9a11-845980a1fd5c"
SQLite = "0aa819cd-b072-5ff4-a722-6bc24af294d9"
SparseArrays = "2f01184e-e22b-5df5-ae63-d93ebab69eaf"
Test = "8dfed614-e22c-5e08-85e1-65c5234f0b40"
TimeZones = "f269a4

**Removed packages and why:**
| Package | Reason removed |
|---|---|
| `Gurobi` | Requires paid license + native lib — can't run in container |
| `Symbolics` | Zero uses in src/ — was a dev dependency left by accident |
| `Debugger`, `Infiltrator`, `Revise` | Dev-only REPL tools, useless in container |
| `Ipopt`, `OhMyREPL` | Unused |

`Plots` kept — used by scripts (mpc_plot.jl), but **not** imported in the FlexOPTi module itself.

### 3b. Module entry point (`FlexOPTi/src/FlexOPTi.jl`)

In [ ]:
with open('FlexOPTi/src/FlexOPTi.jl') as f:
    print(f.read())

**Key change:** Removed `include(plot_state.jl)` from module. Plots stays in scripts only.

**Why?** Module precompile loads all `include`d files. If `plot_state.jl` has `using Plots` and Plots isn't a module dep, precompile fails.

### 3c. Type system (`FlexOPTi/src/core/types.jl`)

Changed `market_id::Int` → `market_country::String` in the `O` (optimization parameters) struct.

In [ ]:
with open('FlexOPTi/src/core/types.jl') as f:
    print(f.read())

**Why?** `market_id=4` was Germany's ID hardcoded from a one-time inspection of the TM registry. IDs can change. Now FC discovers the ID dynamically at runtime by querying `GET /api/market` and matching on `market_location` string.

### 3d. Price fetching (`FlexOPTi/src/core/fetch_prices.jl`)

Core change: dynamic market discovery.

In [ ]:
with open('FlexOPTi/src/core/fetch_prices.jl') as f:
    print(f.read())

**Flow:**
```
_get_prices_dict(o)
  → _get_market_id(tm_base, country)   # GET /api/market → find Germany
  → _load_cache(country)               # check prices_germany.json
  → fetch from TM if cache stale       # GET /api/price?market_id=N
  → _save_cache(country, id, prices)
```

**Try/catch wraps TM call** — if TM is down, FC falls back to cached prices and keeps running.

### 3e. Solver initialization (`FlexOPTi/src/core/mpc_helper.jl`)

Removed all Gurobi branches. Now uses generic solver lookup + HiGHS fallback.

In [ ]:
with open('FlexOPTi/src/core/mpc_helper.jl') as f:
    print(f.read())

### 3f. Settings file (`FlexOPTi/settings.json`)

New file — externalizes parameters that were hardcoded in Julia source.

In [ ]:
import json
with open('FlexOPTi/settings.json') as f:
    settings = json.load(f)

for k, v in settings.items():
    print(f"  {k:20s} = {v}")

**Key params:**
| Key | Meaning |
|---|---|
| `pilot` | Which asset type: `"Ewh"` or `"Montcada"` |
| `solver` | JuMP solver: `"HiGHS"` (open-source MILP) |
| `Hu` | MPC horizon in timesteps (48 × 15min = 12h) |
| `delta_t` | Timestep in seconds (900 = 15 min) |
| `market_country` | Used for dynamic TM market lookup |
| `tm_base_url` | Trading Manager URL inside Docker network |
| `retrain_every_h` | DT retrain cadence (168h = weekly) |

---
## 4. FC Controller Script (`FlexOPTi/scripts/ewh/dr_controller.jl`)

This is the main loop that runs inside the `fc` container.

In [ ]:
with open('FlexOPTi/scripts/ewh/dr_controller.jl') as f:
    print(f.read())

**Main loop logic:**
```
every Δt_s seconds:
  1. check if DT retrain needed (every retrain_every_h)
  2. read latest measurements from DB
  3. read disturbance forecasts from DB
  4. call optimize(pilot, o, dt_file, measurements, forecasts)
  5. write setpoints to DB
  6. expose /status and /health via HTTP on port 8080
```

**Startup shortcut:** if DT JSON already exists on volume, skip retrain. `retrain_every_h=168` means it won't retrain during a demo.

---
## 5. FlexOPTi Dockerfile (`FlexOPTi/Dockerfile`)

Julia containers are slow to build because precompilation is expensive. Structure matters.

In [5]:
with open('FlexOPTi/Dockerfile') as f:
    print(f.read())

FROM julia:1.12

WORKDIR /app

# Copy only Project.toml — Manifest is machine-specific; resolve fresh inside container
COPY Project.toml ./

# Add General registry then resolve + instantiate from Project.toml
RUN julia -e 'using Pkg; Pkg.Registry.add("General")' && \
    julia --project=. -e 'using Pkg; Pkg.resolve(); Pkg.instantiate()'

# Copy source and data
COPY src/              ./src/
COPY data/EWH/inputs/  ./data/EWH/inputs/
COPY data/cache/       ./data/cache/
COPY scripts/          ./scripts/
COPY settings.json     ./
COPY entrypoint.sh     ./

# Create output directories that mpc_main.jl writes to
RUN mkdir -p /app/data/EWH/outputs /app/data/montcada/outputs

# Precompile FlexOPTi so container startup is fast
RUN julia --project=. -e 'using FlexOPTi; println("Precompile OK")'

RUN chmod +x /app/entrypoint.sh

# /dt-volume is the shared volume; DT container writes here; FC reads from here
VOLUME ["/dt-volume"]

ENV DT_FILE=/dt-volume/cold_room_model_continuous.json

ENTRYPOINT 

**Build layer order (why it matters for caching):**
1. `COPY Project.toml` — only the manifest descriptor, not Manifest.toml
2. `RUN julia ... Pkg.resolve(); Pkg.instantiate()` — downloads all packages
3. `COPY src/ scripts/ data/ ...` — source code
4. `RUN julia ... using FlexOPTi` — precompiles the module

**Why not copy Manifest.toml?** Manifest contains machine-specific UUIDs. Copying it into the container causes UUID-not-found errors when the container resolves packages. Fresh resolve inside the container always works.

**Entrypoint:** `/app/entrypoint.sh` — seeds DT volume if empty, then runs dr_controller.jl.

In [6]:
with open('FlexOPTi/entrypoint.sh') as f:
    print(f.read())

#!/bin/bash
set -e

# Seed the shared DT volume with the pre-baked model on first start.
# DT container will overwrite this when it retrains.
if [ ! -f "$DT_FILE" ]; then
    cp /app/data/EWH/inputs/cold_room_model_continuous.json "$DT_FILE"
    echo "[BOOT] Seeded DT volume: $DT_FILE"
fi

exec julia --project=/app /app/scripts/ewh/dr_controller.jl



---
## 6. Control Room (`control_room/`)

Flask dashboard. No frontend framework — plain HTML + JS fetching its own endpoints.

In [ ]:
with open('control_room/main.py') as f:
    print(f.read())

**Endpoints:**
| Endpoint | What it returns |
|---|---|
| `GET /` | HTML dashboard (auto-refresh 15s) |
| `GET /health` | This service's uptime |
| `GET /status` | FC status + last 5 setpoints from DB |
| `GET /services` | Pings fc + dummy-forecast health endpoints |
| `GET /metrics` | Counts + recent rows from all DB tables |
| `GET /setpoints` | Last 50 setpoints |
| `GET /forecasts` | Next 50 forecast rows |

**Try it (with stack running):**

In [1]:
import requests

BASE = "http://localhost:9000"

# Check all service health
r = requests.get(f"{BASE}/services", timeout=10)
import json; print(json.dumps(r.json(), indent=2))

{
  "all_ok": true,
  "services": {
    "control-room": {
      "ok": true,
      "status_code": 200
    },
    "dummy-forecast": {
      "body": {
        "service": "dummy-forecast",
        "status": "ok",
        "uptime_s": 42.4
      },
      "ok": true,
      "status_code": 200
    },
    "fc": {
      "body": "OK",
      "ok": true,
      "status_code": 200
    }
  }
}


In [ ]:
# FC optimization status
r = requests.get(f"{BASE}/status", timeout=10)
print(json.dumps(r.json(), indent=2))

In [ ]:
# Recent setpoints from DB
r = requests.get(f"{BASE}/setpoints", timeout=10)
import pandas as pd
df = pd.DataFrame(r.json())
print(df.to_string())

In [ ]:
# Recent forecasts
r = requests.get(f"{BASE}/forecasts", timeout=10)
df = pd.DataFrame(r.json())
print(df.to_string())

---
## 7. Docker Compose (`docker-compose.ewh.yml`)

Wires all services together.

In [ ]:
with open('docker-compose.ewh.yml') as f:
    print(f.read())

**Network design:**
```
default network (this compose):
  dummy-forecast, fc, control-room, dt-ewh
  → services talk by container name: http://fc:8080, http://dummy-forecast:5000

tm-net (external, Trading Manager compose):
  fc also joins this network
  → fc can reach http://tm-service:8080 for electricity prices
```

**Volume design:**
```
db-data    → /data/bb.db          (dummy-forecast + fc + control-room)
dt-volume  → /dt-volume/          (dt-ewh writes, fc reads)
```

**dt-ewh profile:** `profiles: [retrain]` means it does NOT start with normal `docker compose up`.
Run manually when you want to retrain:
```bash
docker compose -f docker-compose.ewh.yml run --rm dt-ewh
```

---
## 8. Verify the stack is running

In [2]:
import subprocess
result = subprocess.run(
    ['docker', 'compose', '-f', 'docker-compose.ewh.yml', 'ps'],
    capture_output=True, text=True
)
print(result.stdout)

NAME                                   IMAGE                                COMMAND                  SERVICE          CREATED          STATUS          PORTS
flexibility-manager-control-room-1     flexibility-manager-control-room     "python -u main.py"      control-room     12 minutes ago   Up 59 seconds   0.0.0.0:9000->9000/tcp, [::]:9000->9000/tcp
flexibility-manager-dummy-forecast-1   flexibility-manager-dummy-forecast   "python -u dummy_for…"   dummy-forecast   12 minutes ago   Up 58 seconds   0.0.0.0:5000->5000/tcp, [::]:5000->5000/tcp
flexibility-manager-fc-1               flexibility-manager-fc               "/app/entrypoint.sh"     fc               12 minutes ago   Up 58 seconds   0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp



In [3]:
# Tail FC logs (last 30 lines)
result = subprocess.run(
    ['docker', 'logs', '--tail', '30', 'flexibility-manager-fc-1'],
    capture_output=True, text=True
)
print(result.stdout[-3000:])  # last 3000 chars
if result.stderr:
    print("STDERR:", result.stderr[-1000:])


STDERR: Warning: [2026-05-16 13:50:07] No pilot-specific parse_sensors defined. Returning nothing.
└ @ FlexOPTi /app/src/core/parse_sensors.jl:2
[ Info: [2026-05-16 13:50:08] Reading the forecasts from nothing.
[ Info: [2026-05-16 13:50:08] Fetching market prices (country=Germany) from http://tm-service:8080.
[ Info: [2026-05-16 13:50:08] Resolving market prices for horizon t0=2026-05-16T13:50:04.498 UTC, Hu=48, Δt=900.0 s  country=Germany
[ Info: [2026-05-16 13:50:15] Querying TM: market 4, sessions 2026-05-14 → 2026-05-18
[ Info: [2026-05-16 13:50:16]   Found 4 offer session(s)
[ Info: [2026-05-16 13:50:16] Price cache written to /app/data/cache/prices_germany.json (192 slots)
[ Info: [2026-05-16 13:50:16] Market prices fetched live from Trading Manager (market_id=4, 192 slots)
[ Info: [2026-05-16 13:50:16] Price vector aligned: Hu=48 steps, min=0.0001 max=0.0002 EUR/kWh (quality: live)
[ Info: [2026-05-16 13:50:16] Building the constraints.
[ Info: [2026-05-16 13:50:17] Using solve

In [ ]:
# Check FC was actually able to reach TM for prices
result = subprocess.run(
    ['docker', 'network', 'inspect', 'local_tm-net', '--format',
     '{{range .Containers}}{{.Name}}\n{{end}}'],
    capture_output=True, text=True
)
print("Containers on local_tm-net:")
print(result.stdout if result.stdout else "(network not found — TM not running)")

---
## 9. Inspect the SQLite DB directly

In [4]:
# Find the volume mount path
result = subprocess.run(
    ['docker', 'volume', 'inspect', 'flexibility-manager_db-data',
     '--format', '{{.Mountpoint}}'],
    capture_output=True, text=True
)
db_mount = result.stdout.strip()
print("DB volume:", db_mount)

DB volume: /var/lib/docker/volumes/flexibility-manager_db-data/_data


In [5]:
import sqlite3, os

db_path = os.path.join(db_mount, 'bb.db')
print("DB path:", db_path)

if os.path.exists(db_path):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
    for t in tables:
        name = t['name']
        count = conn.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
        print(f"  {name}: {count} rows")
    conn.close()
else:
    print("DB not found — need sudo or run via docker exec")

DB path: /var/lib/docker/volumes/flexibility-manager_db-data/_data/bb.db
DB not found — need sudo or run via docker exec


In [6]:
# Alternative: query via docker exec (no sudo needed)
result = subprocess.run(
    ['docker', 'exec', 'flexibility-manager-dummy-forecast-1',
     'python3', '-c',
     'import sqlite3; conn=sqlite3.connect("/data/bb.db"); '
     '[print(r) for r in conn.execute("SELECT name, (SELECT COUNT(*) FROM " + r[0] + ") FROM sqlite_master WHERE type=\"table\"").fetchall()]'
    ],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr: print(result.stderr)


  File "<string>", line 1
    import sqlite3; conn=sqlite3.connect("/data/bb.db"); [print(r) for r in conn.execute("SELECT name, (SELECT COUNT(*) FROM " + r[0] + ") FROM sqlite_master WHERE type="table"").fetchall()]
                                                                                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
SyntaxError: invalid syntax. Perhaps you forgot a comma?



---
## 10. Summary: What changed and why

| File | Change | Reason |
|---|---|---|
| `db/init.sql` | **NEW** | Central schema for all services |
| `forecasting/dummy/dummy_forecast.py` | **NEW** | Synthetic disturbance forecasts so FC has inputs |
| `forecasting/dummy/Dockerfile` | **NEW** | Container for above |
| `control_room/main.py` | **NEW** | Operator dashboard REST API + HTML |
| `control_room/Dockerfile` | **NEW** | Container for above |
| `control_room/requirements.txt` | **NEW** | Flask + requests |
| `docker-compose.ewh.yml` | **NEW** | Wires all 4 services + volumes + networks |
| `FlexOPTi/Dockerfile` | **NEW** | Julia container, copies only Project.toml (not Manifest) |
| `FlexOPTi/entrypoint.sh` | **NEW** | Seeds DT volume, launches dr_controller.jl |
| `FlexOPTi/settings.json` | **NEW** | Externalizes pilot/solver/horizon/TM URL config |
| `dt/EWH_coldrooms_CSTM/Dockerfile` | **NEW** | R container for Digital Twin retrain |
| `FlexOPTi/Project.toml` | **MODIFIED** | Removed Gurobi, Symbolics, Debugger, Infiltrator, Ipopt, Revise, OhMyREPL |
| `FlexOPTi/src/FlexOPTi.jl` | **MODIFIED** | Removed unused imports; removed plot_state include |
| `FlexOPTi/src/core/types.jl` | **MODIFIED** | `market_id::Int` → `market_country::String` |
| `FlexOPTi/src/core/fetch_prices.jl` | **MODIFIED** | Dynamic market discovery via TM `/api/market` |
| `FlexOPTi/src/core/mpc_helper.jl` | **MODIFIED** | Removed all Gurobi code, generic solver init |
| `FlexOPTi/src/core/fm_optimize.jl` | **MODIFIED** | `market_id=4` → `market_country="Germany"` |
| `FlexOPTi/src/core/plots/plot_state.jl` | **MODIFIED** | Added `using Plots` (for direct script inclusion) |
| `FlexOPTi/scripts/ewh/dr_controller.jl` | **MODIFIED** | Reads settings.json; configurable Δt/Hu/pilot/solver; non-blocking DT check |
| `FlexOPTi/scripts/ewh/mpc_base.jl` | **MODIFIED** | Fixed O() constructor after adding market_country field |
| `README.md` | **REWRITTEN** | Full system description + Docker quick-start + control room examples |
| `.env.example` | **NEW** | Documents all env vars; partners copy → .env |
| `model_dump.lp`, `output.txt` | **DELETED** | Generated artefacts, not source |
| `.docker-documentation.md` | **DELETED** | Replaced by README.md |